In [ ]:
import socket
import threading
import time
import random


# Nama : Alina Rusyda Hariadi  (26051905023) & Gregorius Aldi Bagaskoro (26051905017)
# Tugas 1 - Socket untuk Cluster Computing
# Membuat implementasi sederhana broker-worker dalam komputasi dalam jaringan

# KONFIGURASI DAN PROTOKOL
HOST = '127.0.0.1'

# Fungsi untuk mencari port kosong acak
def find_free_port(start_port, exclude_ports=None):
    if exclude_ports is None:
        exclude_ports = set()
    for port in range(start_port, start_port + 100):
        if port in exclude_ports:
            continue
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
            try:
                s.bind((HOST, port))
                return port
            except OSError:
                continue
    return start_port

# Memilih port dinamis unik secara terjamin
allocated_ports = set()

BROKER_PORT = find_free_port(8000, allocated_ports)
allocated_ports.add(BROKER_PORT)

WORKER_PORTS = []
for base_port in [8010, 8020, 8030]:
    port = find_free_port(base_port, allocated_ports)
    WORKER_PORTS.append(port)
    allocated_ports.add(port)

# 1. WORKER SERVER

def worker_server(worker_id, port):
    server = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    server.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
    try:
        server.bind((HOST, port))
    except OSError as e:
        print(f"[WORKER {worker_id}] Gagal bind di port {port}: {e}")
        return

    server.listen(5)
    print(f"[WORKER {worker_id}] Aktif dan mendengarkan di port {port}")

    while True:
        try:
            client_sock, addr = server.accept()
            data = client_sock.recv(1024).decode('utf-8')
            if not data:
                continue

            # Parsing Pesan
            parts = data.split('|')
            if len(parts) != 3:
                client_sock.sendall("Format Error".encode('utf-8'))
                client_sock.close()
                continue

            app_id, req_type, msg = parts[0], parts[1], parts[2]
            print(f" -> [WORKER {worker_id}] Menerima Request (AppID: {app_id}, Tipe: {req_type})")

            # Simulasi Pemrosesan Aplikasi
            if req_type == 'LONG':
                # Perhitungan kompleks, simulasi dengan sleep
                time.sleep(2)
                response = f"Hasil Kompleks (AppID {app_id}): {msg} diselesaikan Worker {worker_id}"
            elif req_type == 'SHORT':
                # Perhitungan sederhana / Echo
                response = f"Echo (AppID {app_id}): {msg} dari Worker {worker_id}"
            else:
                response = f"Tipe Aplikasi Tidak Dikenal: {req_type}"

            client_sock.sendall(response.encode('utf-8'))
            client_sock.close()
        except Exception as e:
            pass # Abaikan error koneksi untuk simulasi sederhana

# 2. BROKER SERVER (LOAD BALANCER)
class BrokerServer:
    def __init__(self, method='round_robin'):
        # method bisa 'round_robin' (berurutan) atau 'least_connection' (pemerataan)
        self.method = method
        self.rr_counter = 0
        self.worker_loads = {0: 0, 1: 0, 2: 0} # {index_worker: jumlah_request_dilayani}
        self.lock = threading.Lock()

    def start_broker(self):
        server = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        server.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
        try:
            a
        except OSError as e:
            print(f"[BROKER] Gagal bind di port {BROKER_PORT}: {e}")
            return

        server.listen(10)
        print(f"[BROKER] Aktif di port {BROKER_PORT} | Metode Alokasi: {self.method}")

        while True:
            client_sock, addr = server.accept()
            # Handle tiap koneksi client di thread terpisah agar non-blocking
            threading.Thread(target=self.handle_client_request, args=(client_sock,), daemon=True).start()

    def handle_client_request(self, client_sock):
        data = client_sock.recv(1024).decode('utf-8')
        if not data:
            return

        with self.lock:
            # Algoritma Load Balancing
            if self.method == 'round_robin':
                chosen_idx = self.rr_counter % len(WORKER_PORTS)
                self.rr_counter += 1
            elif self.method == 'least_connection':
                # Pilih worker dengan load terkecil untuk pemerataan
                chosen_idx = min(self.worker_loads, key=self.worker_loads.get)

            # Tambah catatan load worker yang dipilih
            self.worker_loads[chosen_idx] += 1
            worker_id = chosen_idx + 1

        print(f"[BROKER] Meneruskan request ke WORKER {worker_id}...")

        # Teruskan ke worker yang terpilih
        try:
            worker_sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
            worker_sock.connect((HOST, WORKER_PORTS[chosen_idx]))
            worker_sock.sendall(data.encode('utf-8'))

            # Tunggu balasan dari worker
            worker_resp = worker_sock.recv(2048)
            worker_sock.close()

            # Kirim kembali balasan ke client awal
            client_sock.sendall(worker_resp)
        except Exception as e:
            client_sock.sendall(f"Worker {worker_id} Down/Error".encode('utf-8'))

        client_sock.close()

# 3. FUNGSI CLIENT
def send_request(app_id, req_type, message):
    client = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    try:
        client.connect((HOST, BROKER_PORT))
        payload = f"{app_id}|{req_type}|{message}"
        client.sendall(payload.encode('utf-8'))
        response = client.recv(2048).decode('utf-8')
        print(f"[CLIENT] Menerima balasan: {response}\n")
    except Exception as e:
        print(f"[CLIENT] Gagal terhubung ke broker: {e}")
    finally:
        client.close()

# 4. BLOK EKSEKUSI SIMULASI (UNTUK COLAB)
if __name__ == "__main__":
    print("=== MEMULAI CLUSTER ===\n")
    print(f"Broker Port: {BROKER_PORT}, Worker Ports: {WORKER_PORTS}")

    # Menjalankan 3 Worker di background thread
    for i in range(3):
        threading.Thread(target=worker_server, args=(i+1, WORKER_PORTS[i]), daemon=True).start()

    time.sleep(1) # Jeda sedikit memastikan worker sudah siap

    # Pilih metode alokasi: 'round_robin' atau 'least_connection'
    metode = 'round_robin'  # SILAKAN UBAH INI UNTUK MENGETES METODE LAIN
    broker = BrokerServer(method=metode)
    threading.Thread(target=broker.start_broker, daemon=True).start()

    time.sleep(1) # Jeda memastikan broker siap
    print("\n=== MENGIRIM REQUEST DARI CLIENT ===\n")

    # Simulasi berbagai request dari Client berurutan
    requests = [
        ("APP-01", "SHORT", "Halo Greg"),
        ("APP-02", "LONG",  "Hitung Matriks Greg"),
        ("APP-03", "SHORT", "Halo Alina"),
        ("APP-04", "LONG",  "Hitung Matriks Alina"),
        ("APP-05", "SHORT", "Halo Greg dan Alina")
    ]

    for req in requests:
        print(f"[CLIENT] Mengirim: AppID {req[0]}, Tipe {req[1]}")
        # Kirim sinkron agar log terminal terlihat rapi sesuai urutan pengiriman
        send_request(req[0], req[1], req[2])

=== MEMULAI CLUSTER ===

Broker Port: 8007, Worker Ports: [8013, 8023, 8032]
[WORKER 1] Aktif dan mendengarkan di port 8013
[WORKER 2] Aktif dan mendengarkan di port 8023
[WORKER 3] Aktif dan mendengarkan di port 8032
[BROKER] Aktif di port 8007 | Metode Alokasi: round_robin

=== MENGIRIM REQUEST DARI CLIENT ===

[CLIENT] Mengirim: AppID APP-01, Tipe SHORT
[BROKER] Meneruskan request ke WORKER 1...
 -> [WORKER 1] Menerima Request (AppID: APP-01, Tipe: SHORT)
[CLIENT] Menerima balasan: Echo (AppID APP-01): Halo Greg dari Worker 1

[CLIENT] Mengirim: AppID APP-02, Tipe LONG
[BROKER] Meneruskan request ke WORKER 2...
 -> [WORKER 2] Menerima Request (AppID: APP-02, Tipe: LONG)
[CLIENT] Menerima balasan: Hasil Kompleks (AppID APP-02): Hitung Matriks Greg diselesaikan Worker 2

[CLIENT] Mengirim: AppID APP-03, Tipe SHORT
[BROKER] Meneruskan request ke WORKER 3...
 -> [WORKER 3] Menerima Request (AppID: APP-03, Tipe: SHORT)
[CLIENT] Menerima balasan: Echo (AppID APP-03): Halo Alina dari Worke